<a href="https://colab.research.google.com/github/samyuktha07-prog/Disaster-Resource-Coordination-Cloud/blob/main/Disaster_Resource_Coordination_Cloud.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:


!pip -q install gradio pandas

import gradio as gr
import pandas as pd
from datetime import datetime
import random
import os

resources = pd.DataFrame([
    {
        "ID": "RES001",
        "Resource": "Drinking Water",
        "Category": "Essential",
        "Quantity": 500,
        "Location": "Chennai",
        "Status": "Available"
    },
    {
        "ID": "RES002",
        "Resource": "Food Packets",
        "Category": "Food",
        "Quantity": 300,
        "Location": "Chennai",
        "Status": "Available"
    },
    {
        "ID": "RES003",
        "Resource": "First Aid Kits",
        "Category": "Medical",
        "Quantity": 100,
        "Location": "Kanchipuram",
        "Status": "Available"
    },
    {
        "ID": "RES004",
        "Resource": "Blankets",
        "Category": "Shelter",
        "Quantity": 250,
        "Location": "Tiruvallur",
        "Status": "Available"
    },
    {
        "ID": "RES005",
        "Resource": "Emergency Tents",
        "Category": "Shelter",
        "Quantity": 50,
        "Location": "Chennai",
        "Status": "Available"
    }
])

requests = pd.DataFrame(columns=[
    "Request ID",
    "Disaster",
    "Location",
    "Resource",
    "Quantity",
    "People Affected",
    "Priority",
    "Status",
    "Created Time"
])

volunteers = pd.DataFrame([
    {
        "Volunteer ID": "VOL001",
        "Name": "Arun",
        "Skill": "Medical",
        "Location": "Chennai",
        "Status": "Available"
    },
    {
        "Volunteer ID": "VOL002",
        "Name": "Priya",
        "Skill": "Rescue",
        "Location": "Tiruvallur",
        "Status": "Available"
    },
    {
        "Volunteer ID": "VOL003",
        "Name": "Karthik",
        "Skill": "Logistics",
        "Location": "Kanchipuram",
        "Status": "Available"
    }
])

disasters = pd.DataFrame(columns=[
    "Incident ID",
    "Disaster Type",
    "Location",
    "Severity",
    "People Affected",
    "Status",
    "Reported Time"
])


def get_dashboard():
    total_resources = len(resources)
    total_quantity = int(resources["Quantity"].sum()) if len(resources) else 0

    pending = 0
    allocated = 0

    if len(requests):
        pending = int((requests["Status"] == "Pending").sum())
        allocated = int((requests["Status"] == "Allocated").sum())

    active_disasters = 0
    if len(disasters):
        active_disasters = int(
            (disasters["Status"] == "Active").sum()
        )

    available_volunteers = int(
        (volunteers["Status"] == "Available").sum()
    )

    return (
        f"### ☁️ DRCC Cloud Dashboard\n\n"
        f"| Metric | Current Value |\n"
        f"|---|---:|\n"
        f"| 🚨 Active Disasters | **{active_disasters}** |\n"
        f"| 📦 Resource Types | **{total_resources}** |\n"
        f"| 📊 Total Resource Units | **{total_quantity}** |\n"
        f"| ⏳ Pending Requests | **{pending}** |\n"
        f"| ✅ Allocated Requests | **{allocated}** |\n"
        f"| 👥 Available Volunteers | **{available_volunteers}** |\n\n"
        f"**System:** Online ☁️  |  **Cloud Status:** Healthy 🟢"
    )


def resource_table():
    return resources.copy()


def request_table():
    return requests.copy()


def volunteer_table():
    return volunteers.copy()


def disaster_table():
    return disasters.copy()

def register_disaster(disaster_type, location, severity, affected):

    global disasters

    if not disaster_type or not location or not affected:
        return "⚠️ Please fill all required fields.", disaster_table(), get_dashboard()

    try:
        affected = int(affected)
    except:
        return "⚠️ People affected must be a number.", disaster_table(), get_dashboard()

    incident_id = "DIS" + str(random.randint(1000, 9999))

    new_data = pd.DataFrame([{
        "Incident ID": incident_id,
        "Disaster Type": disaster_type,
        "Location": location,
        "Severity": severity,
        "People Affected": affected,
        "Status": "Active",
        "Reported Time": datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    }])

    disasters = pd.concat([disasters, new_data], ignore_index=True)

    return (
        f"🚨 **Disaster registered successfully!**\n\n"
        f"Incident ID: **{incident_id}**\n"
        f"Location: **{location}**\n"
        f"Severity: **{severity}**",
        disaster_table(),
        get_dashboard()
    )


def add_resource(name, category, quantity, location):

    global resources

    if not name or not category or not quantity or not location:
        return "⚠️ Please fill all fields.", resource_table(), get_dashboard()

    try:
        quantity = int(quantity)
    except:
        return "⚠️ Quantity must be a number.", resource_table(), get_dashboard()

    resource_id = "RES" + str(random.randint(100, 999))

    new_resource = pd.DataFrame([{
        "ID": resource_id,
        "Resource": name,
        "Category": category,
        "Quantity": quantity,
        "Location": location,
        "Status": "Available"
    }])

    resources = pd.concat(
        [resources, new_resource],
        ignore_index=True
    )

    return (
        f"📦 Resource added successfully!\n\n"
        f"Resource ID: **{resource_id}**",
        resource_table(),
        get_dashboard()
    )


def search_resources(keyword):

    if not keyword:
        return resource_table()

    keyword = keyword.lower()

    result = resources[
        resources.astype(str)
        .apply(
            lambda row: row.str.lower().str.contains(keyword).any(),
            axis=1
        )
    ]

    return result

===

def create_request(
    disaster,
    location,
    resource_name,
    quantity,
    people,
    priority
):

    global requests

    if not resource_name or not location or not quantity or not people:
        return (
            "⚠️ Please fill all required fields.",
            request_table(),
            get_dashboard()
        )

    try:
        quantity = int(quantity)
        people = int(people)
    except:
        return (
            "⚠️ Quantity and people affected must be numbers.",
            request_table(),
            get_dashboard()
        )

    request_id = "REQ" + str(random.randint(1000, 9999))

    new_request = pd.DataFrame([{
        "Request ID": request_id,
        "Disaster": disaster,
        "Location": location,
        "Resource": resource_name,
        "Quantity": quantity,
        "People Affected": people,
        "Priority": priority,
        "Status": "Pending",
        "Created Time": datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    }])

    requests = pd.concat(
        [requests, new_request],
        ignore_index=True
    )

    return (
        f"🚑 **Emergency request created!**\n\n"
        f"Request ID: **{request_id}**\n"
        f"Priority: **{priority}**\n"
        f"Status: **Pending**",
        request_table(),
        get_dashboard()
    )


def allocate_resources():

    global resources
    global requests

    if len(requests) == 0:
        return (
            "ℹ️ No emergency requests available.",
            resource_table(),
            request_table(),
            get_dashboard()
        )

    allocated_count = 0
    messages = []

    # Highest priority first
    priority_order = {
        "Critical": 1,
        "High": 2,
        "Medium": 3,
        "Low": 4
    }

    requests["PriorityRank"] = requests["Priority"].map(
        priority_order
    )

    requests = requests.sort_values(
        by="PriorityRank"
    ).reset_index(drop=True)

    for i in range(len(requests)):

        if requests.loc[i, "Status"] != "Pending":
            continue

        requested_resource = str(
            requests.loc[i, "Resource"]
        ).strip().lower()

        requested_quantity = int(
            requests.loc[i, "Quantity"]
        )

        location = str(
            requests.loc[i, "Location"]
        ).strip().lower()

        found = False

        for j in range(len(resources)):

            resource_name = str(
                resources.loc[j, "Resource"]
            ).strip().lower()

            resource_location = str(
                resources.loc[j, "Location"]
            ).strip().lower()

            available_quantity = int(
                resources.loc[j, "Quantity"]
            )


            if (
                requested_resource == resource_name
                and available_quantity >= requested_quantity
            ):

                resources.loc[j, "Quantity"] = (
                    available_quantity - requested_quantity
                )

                if resources.loc[j, "Quantity"] == 0:
                    resources.loc[j, "Status"] = "Out of Stock"

                requests.loc[i, "Status"] = "Allocated"

                messages.append(
                    f"✅ {requests.loc[i, 'Request ID']} → "
                    f"{requested_quantity} {requests.loc[i, 'Resource']} "
                    f"allocated to {requests.loc[i, 'Location']}"
                )

                allocated_count += 1
                found = True
                break

        if not found:
            messages.append(
                f"⏳ {requests.loc[i, 'Request ID']} → "
                f"Resource unavailable / insufficient quantity"
            )

    requests.drop(
        columns=["PriorityRank"],
        inplace=True
    )

    if not messages:
        messages.append("ℹ️ No pending requests were allocated.")

    return (
        "\n\n".join(messages),
        resource_table(),
        request_table(),
        get_dashboard()
    )



def add_volunteer(name, skill, location):

    global volunteers

    if not name or not skill or not location:
        return "⚠️ Please fill all fields.", volunteer_table()

    volunteer_id = "VOL" + str(random.randint(100, 999))

    new_volunteer = pd.DataFrame([{
        "Volunteer ID": volunteer_id,
        "Name": name,
        "Skill": skill,
        "Location": location,
        "Status": "Available"
    }])

    volunteers = pd.concat(
        [volunteers, new_volunteer],
        ignore_index=True
    )

    return (
        f"👤 Volunteer registered successfully!\n\n"
        f"Volunteer ID: **{volunteer_id}**",
        volunteer_table()
    )


def refresh_all():

    return (
        get_dashboard(),
        resource_table(),
        request_table(),
        volunteer_table(),
        disaster_table()
    )



custom_css = """
body {
    background: #f5f7fb;
}

.gradio-container {
    max-width: 1250px !important;
}

.title {
    text-align: center;
    padding: 20px;
    border-radius: 15px;
}

.card {
    border-radius: 15px;
    padding: 15px;
}

button {
    border-radius: 10px !important;
}
"""

with gr.Blocks(
    title="Disaster Resource Coordination Cloud",
    css=custom_css,
    theme=gr.themes.Soft()
) as app:

    gr.Markdown(
        """
        # ☁️ Disaster Resource Coordination Cloud
        ### 🚨 Cloud-Based Emergency Resource Management System

        **DRCC** is a centralized cloud platform for coordinating
        disaster incidents, emergency resources, requests and volunteers.

        **Architecture:**
        `Users → Cloud Dashboard → Resource Engine → Allocation Engine → Database`
        """
    )

    dashboard = gr.Markdown(
        get_dashboard()
    )

    refresh_btn = gr.Button(
        "🔄 Refresh Cloud Dashboard",
        variant="primary"
    )



    with gr.Tab("🚨 Disaster Management"):

        gr.Markdown(
            "## Register New Disaster Incident"
        )

        with gr.Row():

            with gr.Column():

                disaster_type = gr.Dropdown(
                    [
                        "Flood",
                        "Earthquake",
                        "Cyclone",
                        "Landslide",
                        "Fire",
                        "Tsunami",
                        "Building Collapse",
                        "Other"
                    ],
                    label="Disaster Type"
                )

                disaster_location = gr.Textbox(
                    label="Location",
                    placeholder="Example: Chennai"
                )

            with gr.Column():

                severity = gr.Dropdown(
                    [
                        "Critical",
                        "High",
                        "Medium",
                        "Low"
                    ],
                    label="Severity",
                    value="High"
                )

                affected = gr.Number(
                    label="People Affected",
                    value=100
                )

        register_btn = gr.Button(
            "🚨 Register Disaster",
            variant="primary"
        )

        disaster_message = gr.Markdown()

        disaster_data = gr.Dataframe(
            value=disaster_table(),
            interactive=False
        )

        register_btn.click(
            register_disaster,
            inputs=[
                disaster_type,
                disaster_location,
                severity,
                affected
            ],
            outputs=[
                disaster_message,
                disaster_data,
                dashboard
            ]
        )


    with gr.Tab("📦 Resource Management"):

        gr.Markdown(
            "## Add Emergency Resource"
        )

        with gr.Row():

            resource_name = gr.Textbox(
                label="Resource Name",
                placeholder="Example: Drinking Water"
            )

            resource_category = gr.Dropdown(
                [
                    "Essential",
                    "Food",
                    "Medical",
                    "Shelter",
                    "Rescue",
                    "Communication",
                    "Transport"
                ],
                label="Category"
            )

        with gr.Row():

            resource_quantity = gr.Number(
                label="Quantity",
                value=100
            )

            resource_location = gr.Textbox(
                label="Storage Location",
                placeholder="Example: Chennai"
            )

        add_resource_btn = gr.Button(
            "📦 Add Resource",
            variant="primary"
        )

        resource_message = gr.Markdown()

        resource_data = gr.Dataframe(
            value=resource_table(),
            interactive=False
        )

        add_resource_btn.click(
            add_resource,
            inputs=[
                resource_name,
                resource_category,
                resource_quantity,
                resource_location
            ],
            outputs=[
                resource_message,
                resource_data,
                dashboard
            ]
        )

        gr.Markdown("## 🔍 Search Resources")

        search_box = gr.Textbox(
            label="Search",
            placeholder="Search water, food, Chennai..."
        )

        search_btn = gr.Button(
            "🔍 Search"
        )

        search_result = gr.Dataframe(
            value=resource_table(),
            interactive=False
        )

        search_btn.click(
            search_resources,
            inputs=search_box,
            outputs=search_result
        )


    with gr.Tab("🚑 Emergency Requests"):

        gr.Markdown(
            """
            ## Create Emergency Resource Request

            Requests are automatically prioritized using:
            **Critical → High → Medium → Low**
            """
        )

        with gr.Row():

            request_disaster = gr.Textbox(
                label="Disaster Name",
                placeholder="Example: Chennai Flood"
            )

            request_location = gr.Textbox(
                label="Required Location",
                placeholder="Example: Chennai"
            )

        with gr.Row():

            request_resource = gr.Textbox(
                label="Required Resource",
                placeholder="Example: Drinking Water"
            )

            request_quantity = gr.Number(
                label="Required Quantity",
                value=50
            )

        with gr.Row():

            request_people = gr.Number(
                label="People Affected",
                value=100
            )

            request_priority = gr.Dropdown(
                [
                    "Critical",
                    "High",
                    "Medium",
                    "Low"
                ],
                label="Priority",
                value="High"
            )

        request_btn = gr.Button(
            "🚑 Create Emergency Request",
            variant="primary"
        )

        request_message = gr.Markdown()

        request_data = gr.Dataframe(
            value=request_table(),
            interactive=False
        )

        request_btn.click(
            create_request,
            inputs=[
                request_disaster,
                request_location,
                request_resource,
                request_quantity,
                request_people,
                request_priority
            ],
            outputs=[
                request_message,
                request_data,
                dashboard
            ]
        )

        gr.Markdown(
            "## 🤖 Automatic Cloud Resource Allocation"
        )

        allocate_btn = gr.Button(
            "🤖 Allocate Resources Automatically",
            variant="primary"
        )

        allocation_message = gr.Markdown()

        allocation_resources = gr.Dataframe(
            interactive=False
        )

        allocation_requests = gr.Dataframe(
            interactive=False
        )

        allocate_btn.click(
            allocate_resources,
            outputs=[
                allocation_message,
                allocation_resources,
                allocation_requests,
                dashboard
            ]
        )

    with gr.Tab("👥 Volunteer Management"):

        gr.Markdown(
            "## Register Volunteer"
        )

        with gr.Row():

            volunteer_name = gr.Textbox(
                label="Volunteer Name"
            )

            volunteer_skill = gr.Dropdown(
                [
                    "Medical",
                    "Rescue",
                    "Logistics",
                    "Communication",
                    "Transport",
                    "Food Distribution"
                ],
                label="Skill"
            )

            volunteer_location = gr.Textbox(
                label="Location"
            )

        volunteer_btn = gr.Button(
            "👤 Register Volunteer",
            variant="primary"
        )

        volunteer_message = gr.Markdown()

        volunteer_data = gr.Dataframe(
            value=volunteer_table(),
            interactive=False
        )

        volunteer_btn.click(
            add_volunteer,
            inputs=[
                volunteer_name,
                volunteer_skill,
                volunteer_location
            ],
            outputs=[
                volunteer_message,
                volunteer_data
            ]
        )


    with gr.Tab("☁️ Cloud Database"):

        gr.Markdown(
            """
            ## ☁️ Centralized Cloud Data

            This section represents the centralized cloud database
            used by the Disaster Resource Coordination System.
            """
        )

        gr.Markdown("### 🚨 Disaster Incidents")

        cloud_disasters = gr.Dataframe(
            value=disaster_table(),
            interactive=False
        )

        gr.Markdown("### 📦 Resource Inventory")

        cloud_resources = gr.Dataframe(
            value=resource_table(),
            interactive=False
        )

        gr.Markdown("### 🚑 Emergency Requests")

        cloud_requests = gr.Dataframe(
            value=request_table(),
            interactive=False
        )

        gr.Markdown("### 👥 Volunteers")

        cloud_volunteers = gr.Dataframe(
            value=volunteer_table(),
            interactive=False
        )

        refresh_database = gr.Button(
            "🔄 Sync Cloud Database",
            variant="primary"
        )

        refresh_database.click(
            refresh_all,
            outputs=[
                dashboard,
                cloud_resources,
                cloud_requests,
                cloud_volunteers,
                cloud_disasters
            ]
        )


    with gr.Tab("📚 Project Information"):

        gr.Markdown(
            """
            # 📚 Disaster Resource Coordination Cloud

            ## 🎯 Objective

            To provide a centralized cloud-based platform for managing
            disaster incidents and coordinating emergency resources,
            volunteers and requests.

            ## ☁️ Cloud Computing Concepts

            **1. Centralized Cloud Database**
            Stores disaster, resource, volunteer and request information.

            **2. Resource Pooling**
            Emergency resources are maintained in a common resource pool.

            **3. Automatic Allocation**
            Resources are automatically allocated to pending requests.

            **4. Priority Scheduling**
            Critical requests are processed before lower-priority requests.

            **5. Scalability**
            New disasters, resources and volunteers can be added dynamically.

            **6. Real-Time Dashboard**
            Administrators can monitor the current system state.

            ## 🏗️ System Architecture

            ```
                    ┌─────────────────────┐
                    │       USERS         │
                    │ Citizens / NGOs     │
                    │ Admin / Volunteers  │
                    └──────────┬──────────┘
                               │
                               ▼
                    ┌─────────────────────┐
                    │   CLOUD DASHBOARD   │
                    │      Gradio UI      │
                    └──────────┬──────────┘
                               │
                 ┌─────────────┼─────────────┐
                 ▼             ▼             ▼
          ┌────────────┐ ┌────────────┐ ┌────────────┐
          │ Disaster   │ │ Resource   │ │ Volunteer  │
          │ Management │ │ Management │ │ Management │
          └─────┬──────┘ └─────┬──────┘ └─────┬──────┘
                │              │              │
                └──────────────┼──────────────┘
                               ▼
                    ┌─────────────────────┐
                    │  ALLOCATION ENGINE  │
                    │ Priority Scheduling │
                    └──────────┬──────────┘
                               │
                               ▼
                    ┌─────────────────────┐
                    │   CLOUD DATABASE    │
                    │ Resources / Requests│
                    │ Disasters / Users   │
                    └─────────────────────┘
            ```

            ## 🚀 Future Enhancements

            • Firebase / MongoDB cloud database
            • AWS S3 resource documents
            • Google Maps disaster visualization
            • AI-based disaster prediction
            • SMS / email emergency notifications
            • GPS-based volunteer tracking
            • Multi-cloud disaster backup
            • Role-based authentication
            • IoT sensor integration
            • Real-time disaster alerts

            ## 💡 Technologies

            **Python + Gradio + Pandas + Google Colab**

            """
        )

app.launch(
    share=True,
    debug=False
)

/tmp/ipykernel_567/696332409.py:519: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme, css. Please pass these parameters to launch() instead.
  with gr.Blocks(


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://432f863256e2ff63c9.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
